In [1]:
%reload_ext autotime
import pandas as pd
import os
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
from pprint import pprint
import json5 as json # This is a more forgiving JSON parser that can handle comments, single quotes, and trailing commas
from glob import glob
from tqdm import tqdm
from openai import OpenAI
import base64
files = sorted(glob("../supplements_videos/*.json"))
print(len(files))

7954
time: 904 ms (started: 2026-06-23 14:49:20 +12:00)


In [2]:
from glob import glob
pd.Series([os.path.splitext(f)[1] for f in glob("../supplements_videos/*")]).value_counts()

.json    7954
.mp4     5562
.webm    2263
.mp3       82
.m4a       35
.mkv       10
.part       2
Name: count, dtype: int64

time: 77.8 ms (started: 2026-06-23 14:49:21 +12:00)


In [3]:
print(pd.to_timedelta("00:" + pd.read_csv("../data/supplements.csv").duration).describe())
print("Total video time:", pd.to_timedelta("00:" + pd.read_csv("../data/supplements.csv").duration).sum())

count                     11707
mean     0 days 00:00:58.878875
std      0 days 00:00:40.115888
min             0 days 00:00:01
25%             0 days 00:00:28
50%             0 days 00:00:53
75%             0 days 00:01:21
max             0 days 00:03:00
Name: duration, dtype: object
Total video time: 7 days 23:28:15
time: 138 ms (started: 2026-06-23 14:49:21 +12:00)


In [4]:
pd.read_csv("../data/supplements.csv").sort_values("duration")

,link,duration,title,source,author
2788,https://www.instagram.com/p/DYhBrzBjCNs/,0:01,Hot flashes can be an unpleasant aspect of the menopause ...,Instagram,Tweak India
5161,https://www.instagram.com/reel/DY7iaRBxCcP/,0:02,Natalie Yco | Hi Fam! 🙋🏾‍♀️ Tell me your holy grail ...,Instagram,Natalie Yco
2557,https://www.instagram.com/p/Cvwl00wtZ3r/,0:02,Menopause diet 101 Make sure you're eating enough protein ...,Instagram,Nutritionist | 30:30:30 method
4014,https://www.instagram.com/reel/DU2jYyyCK0L/,0:02,HAIR FOOD | MENO is a menopause specific food ...,Instagram,Oxygen Hair Beaconsfield
2718,https://www.instagram.com/p/DXFnVgZgcJT/,0:02,BONE HEALTH IS MY NEW OBSESSION Many with prolapse ...,Instagram,Kim Vopni - The Vagina Coach
...,...,...,...,...,...
9959,https://www.youtube.com/shorts/GVj6oTzXZT8,3:00,Protein and bones: the connection your doctor doesn't explain ...,YouTube,Javier Giménez - Healthy Fitness
6888,https://www.tiktok.com/@dr.anika.ackerman/video/7192289140925320491,3:00,Boost Your Sex Drive: Female Libido Treatments Explained,TikTok,dr.anika.ackerman
11382,https://www.youtube.com/shorts/zZ7cScqJX_8,3:00,Oral Progesterone for Sleep in Perimenopause and Menopause,YouTube,"Jennifer Roelands, MD Perimenopause, Longevity"
10931,https://www.youtube.com/shorts/kZVpXCmJjQo,3:00,Menopause Supplements That Actually Work,YouTube,fitwithpratham


time: 59.6 ms (started: 2026-06-23 14:49:21 +12:00)


In [5]:
import ffmpeg

def has_audio(video_path):
    try:
        # Probe the video file specifically selecting audio streams
        probe = ffmpeg.probe(video_path, select_streams='a')
        # If the 'streams' list is not empty, an audio track exists
        return len(probe['streams']) > 0
    except ffmpeg.Error as e:
        print(f"Error checking file: {e}")
        return False

time: 6.8 ms (started: 2026-06-23 14:49:21 +12:00)


In [7]:
json_filename = "../supplements_videos/Menopause is a natural phase of life that every woman will experience, yet it often remains misunder [1498212924203766].info.json"
with open(json_filename) as f:
    data = json.load(f)
display(pd.json_normalize(data))
print(data["description"])

,title,description,uploader,uploader_id,thumbnail,view_count,concurrent_view_count,duration,id,formats,timestamp,webpage_url,webpage_url_basename,webpage_url_domain,extractor,extractor_key,thumbnails,display_id,fulltitle,duration_string,upload_date,epoch,format_id,quality,url,protocol,ext,video_ext,audio_ext,dynamic_range,format,_type,http_headers.User-Agent,http_headers.Accept,http_headers.Accept-Language,http_headers.Sec-Fetch-Mode,downloader_options.http_chunk_size,_version.version,_version.release_git_head,_version.repository
0,"Menopause is a natural phase of life that every woman will experience, yet it often remains misu...","Menopause is a natural phase of life that every woman will experience, yet it often remains misu...",HealthPlus Pharmacy,100064758005826,https://scontent-akl1-1.xx.fbcdn.net/v/t15.5256-10/463830374_1253639125769978_350717853282354911...,103,0,86.1,1498212924203766,"[{'format_id': 'sd', 'quality': -3, 'url': 'https://video-akl1-1.xx.fbcdn.net/o1/v/t2/f2/m412/AQ...",1729270847,https://www.facebook.com/healthPluspharmacyng/videos/menopause-is-a-natural-phase-of-life-that-e...,1498212924203766,facebook.com,facebook,Facebook,[{'url': 'https://scontent-akl1-1.xx.fbcdn.net/v/t15.5256-10/463830374_1253639125769978_35071785...,1498212924203766,"Menopause is a natural phase of life that every woman will experience, yet it often remains misu...",1:26,20241018,1782080676,hd,-2,https://video-akl1-1.xx.fbcdn.net/o1/v/t2/f2/m266/AQNZUyimjzE61FRiP8P__JtxXQSlSXx6oFRiN59k59lIvm...,https,mp4,mp4,none,SDR,hd - unknown,video,facebookexternalhit/1.1,"text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8","en-us,en;q=0.5",navigate,262144000,2026.06.09,821bef0f00178916d60dbc86bc0bcb8cc3bae8d5,yt-dlp/yt-dlp


Menopause is a natural phase of life that every woman will experience, yet it often remains misunderstood and under-discussed. 

On this World Menopause Day, we’re shedding light on what menopause really means and raising awareness about the physical and emotional changes that come with it

#HealthPlus #HealthplusPharmacy #HealthBeautyWellness #Health #wellness #Healthcare #Medicine #Supplements #WorldMenopauseDay #HormoneTherapy
time: 172 ms (started: 2026-06-23 14:49:29 +12:00)


In [8]:
def get_prompt(data):
    return f"""This is a video downloaded from {data['extractor']}. Here's the description of the video: {data['description']}.

        The creator of the video is {data.get('channel', 'an unknown channel')} ({data.get('uploader', 'an unknown uploader')})
        It has {data.get('like_count', 'an unknown number of')} likes, {data.get('view_count', 'an unknown number of')} views, and {data.get('comment_count', 'an unknown number of')} comments.
        Taking into account this description, and the video, extract the following information, in JSON format:
        description: What is happening in the video? Provide a detailed description of the actions, context, and any notable elements present in the video.
        transcript: If there is any spoken content in the video, transcribe it into English. If there is no spoken content, indicate "No spoken content".
        tone: What is the overall tone or mood of the video? Is it humorous, serious, educational, emotional, etc.?
        supplements: Does the video mention any supplements, vitamins, or medications? If so, list them. If not, indicate "No supplements mentioned".
        active_ingredients: If any supplements are mentioned, list the active ingredients in those supplements. If no supplements are mentioned, indicate "No active ingredients mentioned".
        symptoms: Does the video mention any specific symptoms, conditions, or health issues? If so, list them. If not, indicate "No symptoms mentioned".
        menopause: Is the video specifically targeting the supplement towards menopause-related symptoms or conditions? Answer True or False.
        language: What language is this video in?
        marketing: Is this video promoting or advertising any product, service, brand, or organization? If so, what is it? Otherwise, indicate "No marketing content".
        job: For the main speaker, what is their job or profession? If it is not mentioned in the video, indicate "No job information". A comma separated string, one or more of the following: therapist, psychologist, pediatrician, doctor, nurse, teacher, professor, social worker, counselor, coach, influencer, content creator?
        sentiment: Does this video recommend a particular supplement, discourage it, or is it neutral? One of negative, neutral or positive
        criticism: If the video is critical of a particular supplement, what are the main criticisms mentioned? 
        alternative_strategies: Does the video mention any alternative strategies to supplements? If so, what are they? A comma separated string. If no alternatives are mentioned, indicate "No alternative strategies mentioned".
        usefulness: Rate the overall usefulness of the video on a scale from 1 to 10, where 1 is not useful at all and 10 is extremely useful.
        misleading: Rate the extent to which the video contains misleading or inaccurate information on a scale from 1 to 10, where 1 is not misleading at all and 10 is extremely misleading.
        quality: Rate the overall quality of the video on a scale from 1 to 10, where 1 is very poor quality and 10 is excellent quality.
        personal_experience: Does the speaker mention any personal experience with supplements? If so, briefly summarize it.

        Do not include comments in your JSON response. Only respond with the JSON object. Make sure the JSON is valid
    """
print(get_prompt(data))

This is a video downloaded from facebook. Here's the description of the video: Menopause is a natural phase of life that every woman will experience, yet it often remains misunderstood and under-discussed. 

On this World Menopause Day, we’re shedding light on what menopause really means and raising awareness about the physical and emotional changes that come with it

#HealthPlus #HealthplusPharmacy #HealthBeautyWellness #Health #wellness #Healthcare #Medicine #Supplements #WorldMenopauseDay #HormoneTherapy.

        The creator of the video is an unknown channel (HealthPlus Pharmacy)
        It has an unknown number of likes, 103 views, and an unknown number of comments.
        Taking into account this description, and the video, extract the following information, in JSON format:
        description: What is happening in the video? Provide a detailed description of the actions, context, and any notable elements present in the video.
        transcript: If there is any spoken content 

In [12]:
client = OpenAI(base_url="https://ai.cer-sandbox.cloud.edu.au/v1", api_key="not needed")

video_filename = json_filename.replace("info.json", data["ext"])
print(video_filename)
with open(video_filename, "rb") as video_file:
    base64_video = "data:video/" + data["ext"] + ";base64," + base64.b64encode(video_file.read()).decode("utf-8")
print(len(base64_video))

reasoning_budget = 16384
grace_period = 1024

response = client.chat.completions.create(
    model="nemotron_3_nano_omni",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "video_url", "video_url": {"url": base64_video}},
                {"type": "text", "text": get_prompt(data)},
            ],
        }
    ],
    max_tokens=20480,
    temperature=0.6,
    top_p=0.95,
    extra_body={
        "thinking_token_budget": reasoning_budget + grace_period,
        "chat_template_kwargs": {
            "enable_thinking": True,
            "reasoning_budget": reasoning_budget,
        },
        "mm_processor_kwargs": {"use_audio_in_video": has_audio(video_filename)},
    },
)
print(response.choices[0].message.reasoning)
pprint(json.loads(response.choices[0].message.content))

../supplements_videos/Menopause is a natural phase of life that every woman will experience, yet it often remains misunder [1498212924203766].mp4
47514962
We need to extract info.

First, description: The video features a pharmacist (woman in white coat) in a pharmacy setting, discussing World Menopause Day, menopause as natural aging, symptoms, and supplement options (Menopause Original with soya, Menopause Plus with flaxseed and green tea, Reload Women's 50+ formula with multivitamins and cranberry). She encourages contacting HealthPlus pharmacy. So description: A pharmacist introduces World Menopause Day and explains menopause, then presents various supplements to manage symptoms, emphasizing that each woman's experience is unique, and invites viewers to seek support.

Transcript: Need to capture spoken content. Let's read from video: "Hi my name is Pharm Daberechi and today is World Menopause Day it's a day to raise awareness about menopause and the available support for women duri